# Diabetes Health-Indicator Risk Prediction — Model Training Run

> **For research and educational purposes only.** The model below is trained on a
> historical public dataset. Its output is not a medical diagnosis and not medical
> advice. See the model card for what this model must **not** be used for.

This notebook **does not train anything** — it reads the artifacts produced by

```
python -m scripts.train_diabetes
```

so every number shown here is the same number in `reports/diabetes/` and
`models/diabetes/metadata.json`. Re-run that script to refresh them.


In [ ]:
import sys, os
# Make the repository root importable when the kernel starts in notebooks/.
_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

import json
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import Image, Markdown, display

pd.set_option("display.max_columns", 60)

DISEASE = "diabetes"
reports = Path("reports") / DISEASE
models = Path("models") / DISEASE

metrics = json.loads((reports / "metrics.json").read_text(encoding="utf-8"))
card = json.loads((models / "metadata.json").read_text(encoding="utf-8"))
shap_global = json.loads((reports / "shap_global.json").read_text(encoding="utf-8"))

print(card["module"])
print("algorithm :", card["model"]["algorithm"])
print("calibration:", card["model"]["calibration"])


## 1. How the data was split

Preprocessing (imputation, scaling, one-hot) happens **inside** the model pipeline, so
it is fitted on training folds only and never sees the held-out rows.


In [ ]:
pd.Series(metrics["split"]).to_frame("value")

## 2. Model comparison

Five candidate pipelines, 5-fold cross-validated on the training set.

In [ ]:
pd.read_csv(reports / "model_comparison.csv")

### Was the winner meaningfully better?

`selection_margin` compares the tuned top-2 against the fold-to-fold standard deviation.
When the gap is smaller than that noise, the leaderboard is **not** a quality ranking.


In [ ]:
pd.Series(metrics["selection_margin"]).to_frame("value")

In [ ]:
display(Markdown((reports / "SELECTION.md").read_text(encoding="utf-8")))

## 3. Calibration

The wrapper is chosen on **out-of-fold training** scores. The test-set table below is
reported for transparency only — choosing on it would make the test metrics optimistic.


In [ ]:
print("selection (training out-of-fold):")
display(pd.DataFrame(metrics["calibration_selection_train_cv"]).set_index("method"))
print("\nreported only (test set) — NOT used to choose:")
display(pd.DataFrame(metrics["calibration_test_comparison"]).set_index("method"))
print("\nchosen:", card["model"]["calibration"])
print(card["model"]["calibration_rationale"])


## 4. Held-out test performance

In [ ]:
rows = {
    "threshold 0.5": metrics["test_set_threshold_0.5"],
    "sensitivity-oriented": {k: v for k, v in metrics["test_set_alternative_threshold"].items()
                             if k != "threshold_rule"},
}
display(pd.DataFrame(rows).loc[
    ["n", "threshold", "roc_auc", "pr_auc", "recall_sensitivity", "specificity",
     "precision", "f1", "accuracy", "brier", "cm_tn", "cm_fp", "cm_fn", "cm_tp"]
])
print(metrics["test_set_alternative_threshold"]["threshold_rule"])


### Which threshold should you read?

0.5 is a convention, not a decision rule. On a low-prevalence problem a well-calibrated
model should rarely output a probability above 0.5 — most people really do have a
below-even chance — so thresholding there gives high specificity and low recall. That
looks like a broken model and is not one.


In [ ]:
note = metrics.get("diagnostics", {}).get("operating_point_note")
display(Markdown(f"> {note}" if note else "_No operating-point note recorded._"))


In [ ]:
for name in ["roc_curve", "pr_curve", "confusion_matrix", "calibration_curve"]:
    png = reports / "figures" / f"{name}.png"
    if png.exists():
        display(Image(filename=str(png)))


### Threshold sweep

How precision, recall and specificity trade off across the decision threshold.

In [ ]:
pd.read_csv(reports / "threshold_sweep.csv")

## 4b. What actually drives this score?

A high ROC-AUC is not by itself evidence that the model learned clinical structure — it
can also come from how the data was collected. The probe below discards **every measured
value** and trains only on *which values are missing*. If that alone reproduces most of
the headline score, the model is largely reading which tests a clinician chose to order.

That is not train/test leakage — the split stays clean — but it is a ceiling on how far
the result transfers to a setting where these tests are ordered routinely.


In [ ]:
diag = metrics.get("diagnostics", {})
print(diag.get("attribution_note", "(no diagnostics recorded)"))
print()

probe = diag.get("missingness_only_probe")
if probe:
    print("missingness-indicators-only model:")
    for k in ("n_indicator_features", "cv_roc_auc_mean", "cv_roc_auc_std",
              "test_roc_auc", "test_pr_auc", "test_accuracy"):
        if k in probe:
            print(f"  {k:<22} {probe[k]}")
    print(f"\n  full model test ROC-AUC {metrics['test_set_threshold_0.5']['roc_auc']:.4f}")
else:
    print("No missing values in this dataset — measurement pattern cannot carry signal.")

print("\ncomplete-case analysis (why it was not used instead):")
print(" ", diag.get("complete_case"))


In [ ]:
miss_csv = reports / "missingness_vs_target.csv"
if miss_csv.exists():
    display(Markdown("**Is a column's missingness itself predictive of the target?**"))
    display(pd.read_csv(miss_csv))
else:
    print("No missing values to analyse.")


### 4c. What could any model achieve on these features?

When two rows carry an identical feature vector but different outcomes, no model that sees
only these features can get both right. That is irreducible error, and it caps accuracy
regardless of the algorithm. Reported so the gap to 1.0 is attributed honestly — and so
that it is *not* used to excuse a model sitting far below the bound.


In [ ]:
dup = diag.get("duplicate_feature_vectors")
display(Markdown(f"> {diag.get('duplicate_conflict_note', '(not recorded)')}"))
if dup:
    display(pd.Series(dup).to_frame("value"))


### 4d. Did SMOTE actually help?

"Use SMOTE for imbalanced data" is repeated far more often than it is checked. Where this
comparison was run, the selected model was cross-validated on identical folds under SMOTE
and under class weighting. SMOTE resamples **inside** each fold's training portion only.

Watch PR-AUC rather than recall: both strategies move the operating point, so recall at a
fixed 0.5 threshold swings hard while the threshold-free ranking barely changes.


In [ ]:
imb_csv = reports / "imbalance_comparison.csv"
if imb_csv.exists():
    display(Markdown(f"> {diag.get('imbalance_note', '')}"))
    display(pd.read_csv(imb_csv))
else:
    print("Imbalance strategies were not compared for this disease.")


## 5. Explainability (SHAP)

SHAP values are computed on the transformed matrix and summed back onto the **original**
feature names. `additivity_max_error` is the largest gap between `base value + sum(SHAP)`
and the model's actual output — near zero means the explanation really does reconstruct
this model rather than being a plausible-looking set of numbers.


In [ ]:
print("explainer:", shap_global["explainer"])
print("additivity max error:", shap_global["additivity_max_error"])
display(pd.DataFrame(shap_global["global_importance"]))


In [ ]:
for name in ["shap_global_importance", "shap_beeswarm"]:
    png = reports / "figures" / f"{name}.png"
    if png.exists():
        display(Image(filename=str(png)))


### A single explained case

Signed contributions for one held-out patient. Positive pushes the model toward the
positive class, negative away from it. This is an explanation of **the model's output**,
not a clinical account of why this person is or is not ill.


In [ ]:
local = shap_global["example_local_explanation"]
print("base value           :", round(local["base_value"], 4))
print("predicted probability:", round(local["predicted_probability"], 4))
display(pd.DataFrame(local["contributions"]))


## 6. Model card

Intended use, out-of-scope use, and the limitations of this model.


In [ ]:
display(Markdown((models / "MODEL_CARD.md").read_text(encoding="utf-8")))